# LLM-as-a-Judge Evaluation of RAG Answers

This notebook scores the answers stored in `rag_evaluation.db` (table `rag_results`) against their ground truths, using a local **Qwen3.5-4B** model as the judge.

For every row, the judge rates five dimensions on a 0–5 scale: **correctness, coverage, consistency, groundedness, relevance**. The scores, the judge's reasons and a pass/fail flag are written to a second SQLite database (`judge_evaluation.db`).

**Pipeline overview**

1. Install dependencies and mount Google Drive
2. Configure paths and judge settings
3. Define the judge prompt, logging helpers and data loader
4. Load the model and define the generation helpers
5. Define `EvaluationPipeline` (prompting, parsing, database, main loop)
6. Run the evaluation. It **resumes automatically** if interrupted: just run the last cell again.

> Use a GPU runtime (the notebook was written for a Colab T4).

## 1. Setup

Install pinned PyTorch (CUDA 13.0 build) plus the libraries used here. If Colab asks for a runtime restart after installation, restart and continue from the next cell.

In [ ]:
!pip install --no-cache-dir \
    torch==2.11.0 \
    torchaudio==2.11.0 \
    torchvision==0.26.0 \
    --index-url https://download.pytorch.org/whl/cu130

!pip install -U transformers accelerate

!pip install -q pandas numpy scikit-learn sentence-transformers bottleneck

Looking in indexes: https://download.pytorch.org/whl/cu130


In [ ]:
from google.colab import drive

# The evaluation DB, the model weights and the judge results all live on Drive.
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2. Imports and configuration

All tunable settings are at the top:

- `EVAL_DB_PATH`: input database with the RAG answers.
- `JUDGE_DB_PATH`: output database. It is kept on **Drive** so checkpoints survive a Colab runtime reset.
- `JUDGE_MAX_NEW_TOKENS`: the judge only outputs a short JSON object, so generation is capped.
- `JUDGE_MAX_ATTEMPTS`: how often a row is re-judged when the model returns invalid JSON.
- `JUDGE_TEMPERATURE`: kept low for stable scoring.

In [ ]:
import contextlib
import json
import os
import re
import sqlite3
import sys
import threading
import time
import warnings
from contextlib import closing
from datetime import datetime, timezone

import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, TextIteratorStreamer

MODEL_ID = "Qwen/Qwen3.5-4B"
MODEL_PATH = "/content/drive/MyDrive/models/Qwen3.5-4B"
EVAL_DB_PATH = "/content/drive/MyDrive/MAI NLP WS26_27/rag_evaluation.db"
# Stored on Drive so that checkpoints survive a Colab runtime reset.
JUDGE_DB_PATH = "/content/drive/MyDrive/MAI NLP WS26_27/judge_evaluation.db"

JUDGE_MAX_NEW_TOKENS = 4096   # the judge only emits a short JSON object
JUDGE_MAX_ATTEMPTS = 3        # retries per row if the JSON is invalid
JUDGE_TEMPERATURE = 0.1       # low temperature -> more stable scoring

## 3. Judge prompt

The system prompt defines the rubric, the rules (including how to treat unanswerable questions), the required JSON format and three worked examples (fully correct, partially correct, incorrect). The judge is asked to return **only** JSON so that the output can be parsed reliably.

In [ ]:
JUDGE_SYSTEM_PROMPT = '''
You are a strict but fair evaluator comparing a candidate answer against a
ground-truth reference. Evaluate meaning, not wording: accept paraphrases,
equivalent explanations, and concise answers.

Score each dimension with an integer from 0 to 5:

- correctness: Information is factually consistent with the ground truth.
- coverage: Essential points, conditions, and recommendations are included.
- consistency: No contradictions of the ground truth (5 = none, 0 = pervasive).
- groundedness: No unsupported, speculative, or fabricated claims (5 = none).
- relevance: The answer addresses the actual question.

Scale: 5 = fully satisfied; 4 = minor gaps; 3 = partially satisfied;
2 = major gaps; 1 = barely satisfied; 0 = not satisfied.

Rules:
- Weigh omissions and errors by importance.
- Use "primary_domain" first; for multi-domain questions, also assess
  coverage of the other listed "domains" as emphasized in the ground truth.
- If "is_unanswerable" is true, a clear acknowledgement of insufficient
  information is correct; penalize fabricated or speculative personal
  information under correctness and groundedness.
- Do not reward keyword matching without semantic correctness.
- Provide "reasons" as a list of concise strings explaining the scores.
  Include one string for each dimension, identifying important errors,
  omissions, contradictions, unsupported claims, or strengths as applicable.

Return ONLY valid JSON (double quotes, no trailing commas, no extra text):

{
  "correctness": 0,
  "coverage": 0,
  "consistency": 0,
  "groundedness": 0,
  "relevance": 0,
  "reasons": [
    "Correctness: ...",
    "Coverage: ...",
    "Consistency: ...",
    "Groundedness: ...",
    "Relevance: ..."
  ]
}

EXAMPLES

QUESTION:
"I am an international student who has started a paid internship.
My insurer says my contributions may change. Why can employment
affect my insurance?"

GROUND TRUTH:
"Employment type, hours, remuneration and whether an internship
is mandatory or voluntary can affect social-insurance treatment.
Student status alone does not settle every case. Give the insurer
and employer accurate study and employment details and request
a written classification before estimating take-home pay."

Example 1 — Fully correct answer

CANDIDATE ANSWER:
"Your insurance contributions can depend on the type of employment,
working hours, and salary. Mandatory and voluntary internships may
also be treated differently. Being a student does not automatically
determine your insurance status, so you should provide your insurer
and employer with accurate details and request written confirmation
of your classification."

YOUR EXPECTED OUTPUT:
{
  "correctness": 5,
  "coverage": 5,
  "consistency": 5,
  "groundedness": 5,
  "relevance": 5,
  "reasons": [
    "Correctness: The explanation aligns with the reference.",
    "Coverage: It includes the key factors and recommended actions.",
    "Consistency: It does not contradict the reference.",
    "Groundedness: It makes no unsupported claims.",
    "Relevance: It directly answers the question."
  ]
}

Example 2 — Partially correct answer

CANDIDATE ANSWER:
"Your insurance contributions may change because you are earning
money through employment. You should contact your insurer to
check whether your contributions will increase."

YOUR EXPECTED OUTPUT:
{
  "correctness": 3,
  "coverage": 1,
  "consistency": 5,
  "groundedness": 5,
  "relevance": 3,
  "reasons": [
    "Correctness: It gives a plausible but incomplete explanation.",
    "Coverage: It omits most classification factors and recommended steps.",
    "Consistency: It does not contradict the reference.",
    "Groundedness: Its claims are supported by the reference.",
    "Relevance: It addresses the question but gives limited explanation."
  ]
}

Example 3 — Incorrect answer

CANDIDATE ANSWER:
"Your insurance contributions cannot change because you are a
student. All international students are automatically exempt
from social-insurance contributions, regardless of their salary,
working hours, or type of internship."

YOUR EXPECTED OUTPUT:
{
  "correctness": 0,
  "coverage": 0,
  "consistency": 0,
  "groundedness": 0,
  "relevance": 1,
  "reasons": [
    "Correctness: The claimed automatic exemption is false.",
    "Coverage: It omits the essential explanation and recommendations.",
    "Consistency: It directly contradicts the reference.",
    "Groundedness: Its blanket exemption claim is unsupported.",
    "Relevance: It addresses the topic but fails to answer correctly."
  ]
}
'''.strip()

In [ ]:
METRICS = (
    "correctness",
    "coverage",
    "consistency",
    "groundedness",
    "relevance",
)

## 4. Helper functions

- `log()` prints timestamped, immediately flushed messages, which is what gives live feedback in the terminal or notebook output.
- `fmt_duration()` turns seconds into a readable string such as `3m12s` for the progress and ETA lines.

In [ ]:
def log(message):
    """Timestamped, flushed terminal output."""
    print(f"[{datetime.now().strftime('%H:%M:%S')}] {message}", flush=True)


def fmt_duration(seconds):
    seconds = int(max(seconds, 0))
    hours, rem = divmod(seconds, 3600)
    minutes, secs = divmod(rem, 60)
    if hours:
        return f"{hours}h{minutes:02d}m{secs:02d}s"
    if minutes:
        return f"{minutes}m{secs:02d}s"
    return f"{secs}s"

## 5. Loading the evaluation data

Reads the RAG answers from SQLite into a DataFrame. Column and table names are quoted safely. Each row needs a unique `id`, because the id is what makes resuming possible.

In [ ]:
def load_eval_df(database_path: str = EVAL_DB_PATH, table_name: str = "rag_results"):
    columns = [
        "id",
        "type",
        "domains",
        "primary_domain",
        "question",
        "ground_truth",
        "response",
        "fetched_context",
        "is_unanswerable",
        "metadata",
        "created_at",
    ]

    # Quote identifiers safely for SQLite.
    def quote_identifier(name):
        return '"' + name.replace('"', '""') + '"'

    column_list = ", ".join(quote_identifier(column) for column in columns)
    query = f"SELECT {column_list} FROM {quote_identifier(table_name)}"

    connection = sqlite3.connect(database_path)
    try:
        eval_df = pd.read_sql_query(query, connection)
    finally:
        connection.close()

    return eval_df

## 6. Loading the judge model

- The tokenizer is read from the local model folder when available (falls back to the Hub otherwise).
- The dtype is chosen from the GPU's compute capability: **bfloat16 only on Ampere or newer**, otherwise float16. A T4 has no native bf16, and `torch.cuda.is_bf16_supported()` can still report `True` there, which would make inference slow.
- Loading weights from Drive can take several minutes; the log lines say so while it runs.

In [ ]:
def load_model(model_id=MODEL_ID, model_path=MODEL_PATH):
    """Load Qwen3.5-4B."""

    if not torch.cuda.is_available():
        raise SystemExit("No CUDA GPU found.")

    log(f"GPU: {torch.cuda.get_device_name(0)}")

    # Prefer the local copy; fall back to the Hub only if it is missing.
    tokenizer_source = model_path if os.path.isdir(model_path) else model_id
    log(f"Loading tokenizer from {tokenizer_source} ...")
    tokenizer = AutoTokenizer.from_pretrained(
        tokenizer_source,
        trust_remote_code=True,
    )

    # torch.cuda.is_bf16_supported() can report True on GPUs without native
    # bf16 (e.g. the Colab T4, which is slow when emulating it), so check the
    # compute capability instead (bf16 needs Ampere = 8.x or newer).
    major, _ = torch.cuda.get_device_capability(0)
    dtype = torch.bfloat16 if major >= 8 else torch.float16
    log(f"Loading model weights from {model_path} (dtype={dtype}); "
        "reading from Google Drive can take several minutes ...")

    start = time.time()
    model = AutoModelForCausalLM.from_pretrained(
        model_path,
        dtype=dtype,
        device_map="auto",
        local_files_only=True,
        trust_remote_code=True,
    )
    model.eval()
    log(f"Model loaded in {fmt_duration(time.time() - start)}.")

    return tokenizer, model

## 7. Text generation

`generate()` runs `model.generate` in a background thread and consumes tokens from a `TextIteratorStreamer`, optionally printing them live (`verbose=True`). Details worth knowing:

- Errors raised in the worker thread are captured and **re-raised** in the caller (otherwise the streamer would just hang).
- The streamer keeps special tokens, so leftovers like `<|im_end|>` are stripped afterwards. They would otherwise break JSON parsing.
- If thinking mode is on, the text before `</think>` is returned separately from the answer.

In [ ]:
_SPECIAL_TOKEN_RE = re.compile(r"<\|[^|>]*\|>")


def generate(
    tokenizer,
    model,
    messages,
    user_temperature=0.5,
    think=True,
    max_new_tokens=64000,
    verbose=False,
):
    """Generate response incrementally using token streaming."""

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")

        input_text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=think,
        )

        inputs = tokenizer(
            [input_text],
            return_tensors="pt",
        ).to(model.device)

    if think:
        sampling = dict(
            temperature=max(user_temperature - 0.1, 0.01),
            top_p=0.95,
            top_k=20,
        )
    else:
        sampling = dict(
            temperature=max(user_temperature, 0.01),
            top_p=0.8,
            top_k=20,
        )

    streamer = TextIteratorStreamer(
        tokenizer,
        skip_prompt=True,
        skip_special_tokens=False,
        timeout=300,
    )

    generation_kwargs = dict(
        **inputs,
        streamer=streamer,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
        **sampling,
    )

    errors = []

    def generate_silently():
        try:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                with open(os.devnull, "w") as devnull:
                    with contextlib.redirect_stderr(devnull):
                        model.generate(**generation_kwargs)
        except BaseException as exc:  # surface errors from the worker thread
            errors.append(exc)
            streamer.text_queue.put(streamer.stop_signal)

    thread = threading.Thread(target=generate_silently, daemon=True)
    thread.start()

    thinking = ""
    answer = ""
    in_thinking = think

    if verbose:
        print("\nGenerating...\n")

    for text in streamer:
        if not text:
            continue

        if in_thinking and "</think>" in text:
            before, after = text.split("</think>", 1)
            thinking += before
            in_thinking = False
            answer += after
            if verbose:
                print("\n\n--- Answer ---\n" + after, end="", flush=True)
        elif in_thinking:
            thinking += text
            if verbose:
                print(text, end="", flush=True)
        else:
            answer += text
            if verbose:
                print(text, end="", flush=True)

    thread.join()

    if errors:
        raise errors[0]

    if verbose:
        print("\n\nGeneration complete.")

    # skip_special_tokens=False leaves tokens such as <|im_end|> in the text;
    # they would break JSON parsing, so remove them.
    thinking = _SPECIAL_TOKEN_RE.sub("", thinking)
    answer = _SPECIAL_TOKEN_RE.sub("", answer)

    return thinking.strip(), answer.strip()

`ask()` is a thin wrapper that builds the system + user messages and returns `(thinking, answer)`.

In [ ]:
def ask(
    tokenizer,
    model,
    system_prompt: str,
    question: str,
    think=False,
    max_new_tokens=64000,
    user_temperature=0.5,
    verbose=True,
):
    """Send question and stream model output. Returns (thinking, answer)."""

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": question},
    ]

    return generate(
        tokenizer=tokenizer,
        model=model,
        messages=messages,
        user_temperature=user_temperature,
        think=think,
        max_new_tokens=max_new_tokens,
        verbose=verbose,
    )

## 8. The evaluation pipeline

`EvaluationPipeline` bundles everything needed for a run:

| Part | Purpose |
|---|---|
| `make_compare_prompt`, `judge_candidate_answer`, `judge_dataframe_row` | Build the prompt for a row and query the judge; invalid JSON is retried up to `JUDGE_MAX_ATTEMPTS` times |
| `parse_judge_json` | Strictly validates the output: five integer scores in 0–5 and a list of reasons |
| `create_judge_database`, `insert_result`, `update_results` | SQLite storage. A `question_id` column plus a unique index identify each result |
| `get_completed_ids` | Returns the ids already stored for the current run configuration |
| `conduct_judge_evaluation` | Main loop with progress output and resume logic |

**How resuming works:** every result is committed to the database right after it is judged. When the loop starts, it looks up which `question_id`s already exist for the same `rag_algorithm`, `top_k` and `reranking_method` and skips them. Rows that failed are not stored, so they are retried on the next run. Ctrl+C / "Interrupt execution" stops cleanly after the current row.

**Terminal feedback:** per-row scores, PASS/FAIL, duration, elapsed time and ETA, running means every 10 rows, and a summary at the end.

In [ ]:
class EvaluationPipeline:
    """Evaluate candidate answers with a judge LLM and store results in SQLite."""

    METRICS = METRICS

    UPDATABLE_COLUMNS = {
        "rag_algorithm",
        "top_k",
        "reranking_method",
        "selected_chunks",
        "correctness",
        "coverage",
        "consistency",
        "groundedness",
        "relevance",
        "reasons",
        "success",
        "created_at",
    }

    def __init__(
        self,
        eval_df=None,
        system_prompt=None,
        db_path=JUDGE_DB_PATH,
    ):
        self.tokenizer, self.model = load_model()
        self.system_prompt = (
            JUDGE_SYSTEM_PROMPT if system_prompt is None else system_prompt
        )
        self.evaluation_dataframe = (
            pd.DataFrame() if eval_df is None else eval_df
        )
        self.eval_metrics = self.METRICS
        self.db_path = db_path

    # -- prompting ----------------------------------------------------------
    @staticmethod
    def make_compare_prompt(user_input, ground_truth, candidate_answer, metadata):
        """Build a comparison prompt for one candidate answer."""
        metadata_json = json.dumps(
            metadata, ensure_ascii=False, indent=2, default=str
        )

        return f"""
Metadata:
{metadata_json}

USER INPUT:
{user_input}

GROUND TRUTH:
{ground_truth}

CANDIDATE LLM ANSWER:
{candidate_answer}

Compare the candidate with the ground truth using the rubric.
Return only the required JSON.
""".strip()

    def judge_candidate_answer(
        self,
        user_input,
        ground_truth,
        candidate_answer,
        metadata=None,
        verbose=False,
    ):
        """
        Evaluate one answer.

        Returns (scores, reasons_json). If the judge output is not valid JSON,
        the call is retried up to JUDGE_MAX_ATTEMPTS times. If verbose=True the
        judge output is streamed to the terminal.
        """
        prompt = self.make_compare_prompt(
            user_input=user_input,
            ground_truth=ground_truth,
            candidate_answer=candidate_answer,
            metadata=metadata or {},
        )

        last_error = None
        for attempt in range(1, JUDGE_MAX_ATTEMPTS + 1):
            _, raw_verdict = ask(
                tokenizer=self.tokenizer,
                model=self.model,
                system_prompt=self.system_prompt,
                question=prompt,
                think=False,
                max_new_tokens=JUDGE_MAX_NEW_TOKENS,
                user_temperature=JUDGE_TEMPERATURE,
                verbose=verbose,
            )
            try:
                return self.parse_judge_json(raw_verdict)
            except (ValueError, TypeError) as exc:
                last_error = exc
                log(f"    invalid judge output (attempt {attempt}/"
                    f"{JUDGE_MAX_ATTEMPTS}): {exc}")

        raise ValueError(
            f"Judge failed after {JUDGE_MAX_ATTEMPTS} attempts: {last_error}"
        )

    def judge_dataframe_row(self, row, verbose=False):
        """Evaluate a dataframe row."""
        metadata = {
            "id": str(row.get("id", "")),
            "type": row.get("type"),
            "domains": row.get("domains"),
            "primary_domain": row.get("primary_domain"),
            "is_unanswerable": row.get("is_unanswerable", False),
            "fetched_context": row.get("fetched_context"),
        }

        return self.judge_candidate_answer(
            user_input=row.get("question", ""),  # column is named "question"
            ground_truth=row["ground_truth"],
            candidate_answer=row["response"],
            metadata=metadata,
            verbose=verbose,
        )

    def parse_judge_json(self, raw):
        """Validate judge output and return scores and JSON-encoded reasons."""
        if not isinstance(raw, str):
            raise TypeError("Judge response must be a string.")

        raw = _SPECIAL_TOKEN_RE.sub("", raw)
        raw = re.sub(
            r"^```(?:json)?\s*|\s*```$",
            "",
            raw.strip(),
            flags=re.IGNORECASE,
        )

        # Tolerate stray text around the JSON object.
        start, end = raw.find("{"), raw.rfind("}")
        if start == -1 or end <= start:
            raise ValueError("No JSON object found in judge response.")

        try:
            scores_json = json.loads(raw[start:end + 1])
        except json.JSONDecodeError as exc:
            raise ValueError(f"Invalid JSON: {exc}") from exc

        if not isinstance(scores_json, dict):
            raise ValueError("Judge response must be a JSON object.")

        scores = {}
        for metric in self.eval_metrics:
            score = scores_json.get(metric)

            if isinstance(score, float) and score.is_integer():
                score = int(score)

            if isinstance(score, bool) or not isinstance(score, int):
                raise ValueError(f"{metric} must be an integer between 0 and 5.")

            if not 0 <= score <= 5:
                raise ValueError(f"{metric} must be an integer between 0 and 5.")

            scores[metric] = score

        reasons = scores_json.get("reasons")
        if not isinstance(reasons, list) or not all(
            isinstance(reason, str) for reason in reasons
        ):
            raise ValueError("reasons must be a list of strings.")

        return scores, json.dumps(reasons, ensure_ascii=False)

    # -- database -----------------------------------------------------------
    def connect_db(self):
        db_dir = os.path.dirname(self.db_path)
        if db_dir:
            os.makedirs(db_dir, exist_ok=True)
        conn = sqlite3.connect(self.db_path)
        conn.row_factory = sqlite3.Row
        conn.execute("PRAGMA foreign_keys = ON")
        return conn

    def create_judge_database(self):
        """Create the results table if it does not already exist.

        A question_id column identifies which evaluated row a result belongs
        to; it is what makes resuming after an interruption possible.
        """
        schema = """
        CREATE TABLE IF NOT EXISTS evaluation_results (
            id                  INTEGER PRIMARY KEY,
            question_id         TEXT,
            rag_algorithm       TEXT NOT NULL,
            top_k               INTEGER NOT NULL CHECK (top_k > 0),
            reranking_method    TEXT NOT NULL,
            selected_chunks     TEXT NOT NULL,
            correctness         INTEGER NOT NULL CHECK (correctness BETWEEN 0 AND 5),
            coverage            INTEGER NOT NULL CHECK (coverage BETWEEN 0 AND 5),
            consistency         INTEGER NOT NULL CHECK (consistency BETWEEN 0 AND 5),
            groundedness        INTEGER NOT NULL CHECK (groundedness BETWEEN 0 AND 5),
            relevance           INTEGER NOT NULL CHECK (relevance BETWEEN 0 AND 5),
            reasons             TEXT NOT NULL,
            success             INTEGER NOT NULL CHECK (success IN (0, 1)),
            created_at          TEXT NOT NULL
        )
        """

        with closing(self.connect_db()) as conn:
            with conn:
                conn.execute(schema)

            columns = {
                row["name"]
                for row in conn.execute("PRAGMA table_info(evaluation_results)")
            }
            if "selected_chunks" not in columns or "fetched_context" in columns:
                raise ValueError(
                    "The existing evaluation_results table uses an incompatible "
                    "schema. Migrate it or use a new database file."
                )

            # Older versions of this table had no question_id column.
            if "question_id" not in columns:
                with conn:
                    conn.execute(
                        "ALTER TABLE evaluation_results ADD COLUMN question_id TEXT"
                    )

            with conn:
                conn.execute(
                    """
                    CREATE UNIQUE INDEX IF NOT EXISTS ux_eval_run_question
                    ON evaluation_results (
                        question_id, rag_algorithm, top_k, reranking_method
                    )
                    """
                )

    def get_completed_ids(self, rag_algorithm, top_k, reranking_method):
        """question_ids already stored for this run configuration."""
        with closing(self.connect_db()) as conn:
            rows = conn.execute(
                """
                SELECT question_id FROM evaluation_results
                WHERE rag_algorithm = ? AND top_k = ? AND reranking_method = ?
                  AND question_id IS NOT NULL
                """,
                (rag_algorithm, top_k, reranking_method),
            ).fetchall()
        return {row["question_id"] for row in rows}

    @staticmethod
    def _as_json_text(value):
        """Keep strings as-is; serialize other values as JSON text."""
        if isinstance(value, str):
            return value
        return json.dumps(value, ensure_ascii=False, default=str)

    def insert_result(
        self,
        rag_algorithm,
        top_k,
        reranking_method,
        selected_chunks,
        correctness,
        coverage,
        consistency,
        groundedness,
        relevance,
        reasons,
        success,
        created_at=None,
        question_id=None,
    ):
        """Insert one evaluation result and return its generated ID."""
        query = """
        INSERT INTO evaluation_results (
            question_id, rag_algorithm, top_k, reranking_method, selected_chunks,
            correctness, coverage, consistency, groundedness, relevance,
            reasons, success, created_at
        ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        """

        if created_at is None:
            created_at = datetime.now(timezone.utc).isoformat()

        values = (
            None if question_id is None else str(question_id),
            rag_algorithm,
            top_k,
            reranking_method,
            self._as_json_text(selected_chunks),
            correctness,
            coverage,
            consistency,
            groundedness,
            relevance,
            self._as_json_text(reasons),
            int(success),
            created_at,
        )

        with closing(self.connect_db()) as conn:
            with conn:  # commits immediately -> this is the checkpoint
                cursor = conn.execute(query, values)
                return cursor.lastrowid

    def update_results(self, values, *, result_id=None):
        """
        Update allowed columns in one row or, if result_id is None, all rows.

        Returns the number of rows matched.
        """
        if not values:
            raise ValueError("Provide at least one value to update.")

        invalid_columns = set(values) - self.UPDATABLE_COLUMNS
        if invalid_columns:
            raise ValueError(f"Invalid columns: {sorted(invalid_columns)}")

        values = dict(values)
        for column in ("selected_chunks", "reasons"):
            if column in values:
                values[column] = self._as_json_text(values[column])

        if "success" in values and isinstance(values["success"], bool):
            values["success"] = int(values["success"])

        columns = list(values)
        assignments = ", ".join(f"{column} = ?" for column in columns)
        parameters = [values[column] for column in columns]

        query = f"UPDATE evaluation_results SET {assignments}"
        if result_id is not None:
            query += " WHERE id = ?"
            parameters.append(result_id)

        with closing(self.connect_db()) as conn:
            with conn:
                cursor = conn.execute(query, parameters)
                return cursor.rowcount

    # -- main loop ----------------------------------------------------------
    def conduct_judge_evaluation(
        self,
        rag_algorithm="BM25",
        top_k=15,
        reranking_method="none",
        threshold=60,
    ):
        """Evaluate all dataframe rows (resuming if possible); return number stored."""
        if isinstance(top_k, bool) or not isinstance(top_k, int) or top_k <= 0:
            raise ValueError("top_k must be a positive integer.")

        if not 0 <= threshold <= 100:
            raise ValueError("threshold must be between 0 and 100.")

        required_columns = {"id", "ground_truth", "response", "fetched_context"}
        missing = required_columns - set(self.evaluation_dataframe.columns)
        if missing:
            raise ValueError(f"Missing dataframe columns: {sorted(missing)}")

        if reranking_method is None:
            reranking_method = "none"

        df = self.evaluation_dataframe
        total = len(df)
        if df["id"].astype(str).duplicated().any():
            raise ValueError(
                "Column 'id' contains duplicates; ids are needed to resume."
            )

        self.create_judge_database()
        completed = self.get_completed_ids(rag_algorithm, top_k, reranking_method)
        already_done = sum(1 for qid in df["id"].astype(str) if qid in completed)

        log("=" * 70)
        log(f"Run: rag_algorithm={rag_algorithm!r}, top_k={top_k}, "
            f"reranking={reranking_method!r}, threshold={threshold}")
        log(f"Database: {self.db_path}")
        log(f"{total} rows total, {already_done} already evaluated "
            f"-> {total - already_done} to do")
        if already_done:
            log("Resuming from the last checkpoint.")
        log("=" * 70)

        stored = 0
        failed = []
        passed = 0
        score_sums = {metric: 0 for metric in self.eval_metrics}
        run_start = time.time()
        position = 0

        try:
            for _, row in df.iterrows():
                position += 1
                question_id = str(row["id"])

                if question_id in completed:
                    continue

                row_start = time.time()
                log(f"[{position}/{total}] id={question_id} "
                    f"({row.get('primary_domain', '?')}) judging ...")

                try:
                    scores, reasons_json = self.judge_dataframe_row(row)
                except KeyboardInterrupt:
                    raise
                except Exception as exc:
                    failed.append(question_id)
                    log(f"[{position}/{total}] id={question_id} FAILED: {exc} "
                        "(will be retried on the next run)")
                    continue

                avg_score = sum(scores.values()) / len(scores)
                success = avg_score * 20 >= threshold

                self.insert_result(
                    question_id=question_id,
                    rag_algorithm=rag_algorithm,
                    top_k=top_k,
                    reranking_method=reranking_method,
                    selected_chunks=row["fetched_context"],
                    correctness=scores["correctness"],
                    coverage=scores["coverage"],
                    consistency=scores["consistency"],
                    groundedness=scores["groundedness"],
                    relevance=scores["relevance"],
                    reasons=reasons_json,
                    success=success,
                )
                stored += 1
                passed += int(success)
                for metric in self.eval_metrics:
                    score_sums[metric] += scores[metric]

                # Progress / ETA based on rows judged in this session.
                elapsed = time.time() - run_start
                remaining = total - already_done - stored - len(failed)
                eta = elapsed / stored * remaining if stored else 0
                score_text = " ".join(
                    f"{m[:4]}={scores[m]}" for m in self.eval_metrics
                )
                log(f"[{position}/{total}] id={question_id} saved | {score_text} | "
                    f"avg={avg_score:.2f} | {'PASS' if success else 'FAIL'} | "
                    f"{fmt_duration(time.time() - row_start)} | "
                    f"elapsed {fmt_duration(elapsed)} | ETA {fmt_duration(eta)}")

                if stored % 10 == 0:
                    means = ", ".join(
                        f"{m}={score_sums[m] / stored:.2f}"
                        for m in self.eval_metrics
                    )
                    log(f"    running means ({stored} rows this session): {means} "
                        f"| pass rate {passed / stored:.0%}")

        except KeyboardInterrupt:
            log("Interrupted by user. Progress up to the last finished row is saved; "
                "run again to resume.")

        done_now = already_done + stored
        log("=" * 70)
        log(f"Finished session: {stored} newly stored, {len(failed)} failed, "
            f"{done_now}/{total} complete overall "
            f"(session time {fmt_duration(time.time() - run_start)}).")
        if stored:
            means = ", ".join(
                f"{m}={score_sums[m] / stored:.2f}" for m in self.eval_metrics
            )
            log(f"Session means: {means} | pass rate {passed / stored:.0%}")
        if failed:
            log(f"Failed ids (rerun to retry): {failed}")
        log("=" * 70)

        return stored

## 9. Run the evaluation

`main()` loads the data, builds the pipeline (which loads the model) and judges every row. A row counts as a **success** when its average score, scaled to 0–100, reaches `threshold` (60 here).

If the run is interrupted (timeout, disconnect, manual stop), simply **run this cell again with the same settings**. Already judged rows are skipped. If you change `rag_algorithm`, `top_k` or `reranking_method`, it is treated as a new run and everything is evaluated again.

In [ ]:
def main():
    log("Loading evaluation data ...")
    eval_df = load_eval_df()
    log(f"Loaded {len(eval_df)} rows.")

    pipeline = EvaluationPipeline(eval_df=eval_df)

    pipeline.conduct_judge_evaluation(
        rag_algorithm="BM25, Dense Vector Search",
        top_k=15,
        reranking_method="RRF",
        threshold=60,
    )

    print("Evaluation completed.")


if __name__ == "__main__":
    main()

[20:55:53] Loading evaluation data ...
[20:55:53] Loaded 100 rows.
[20:55:53] GPU: Tesla T4
[20:55:53] Loading tokenizer from /content/drive/MyDrive/models/Qwen3.5-4B ...
[20:55:55] Loading model weights from /content/drive/MyDrive/models/Qwen3.5-4B (dtype=torch.float16); reading from Google Drive can take several minutes ...


Loading weights:   0%|          | 0/426 [00:00<?, ?it/s]

[20:56:47] Model loaded in 51s.
[20:56:47] ======================================================================
[20:56:47] Run: rag_algorithm='BM25, Dense Vector Search', top_k=15, reranking='RRF', threshold=60
[20:56:47] Database: /content/drive/MyDrive/MAI NLP WS26_27/judge_evaluation.db
[20:56:47] 100 rows total, 0 already evaluated -> 100 to do
[20:56:47] ======================================================================
[20:56:47] [1/100] id=L01 (Starting studies) judging ...


[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.
[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `fused_recurrent_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


[20:57:13] [1/100] id=L01 saved | corr=4 cove=3 cons=5 grou=5 rele=5 | avg=4.40 | PASS | 25s | elapsed 25s | ETA 41m53s
[20:57:13] [2/100] id=L02 (Starting studies) judging ...
[20:57:33] [2/100] id=L02 saved | corr=5 cove=5 cons=5 grou=5 rele=5 | avg=5.00 | PASS | 20s | elapsed 45s | ETA 37m07s
[20:57:33] [3/100] id=L03 (Starting studies) judging ...
[20:57:57] [3/100] id=L03 saved | corr=5 cove=5 cons=5 grou=5 rele=5 | avg=5.00 | PASS | 24s | elapsed 1m09s | ETA 37m39s
[20:57:57] [4/100] id=L04 (Starting studies) judging ...
[20:58:18] [4/100] id=L04 saved | corr=5 cove=5 cons=5 grou=5 rele=5 | avg=5.00 | PASS | 20s | elapsed 1m30s | ETA 36m18s
[20:58:18] [5/100] id=L05 (Working and Finances) judging ...
[20:58:38] [5/100] id=L05 saved | corr=5 cove=5 cons=5 grou=5 rele=5 | avg=5.00 | PASS | 19s | elapsed 1m50s | ETA 34m59s
[20:58:38] [6/100] id=L06 (Working and Finances) judging ...
[20:58:59] [6/100] id=L06 saved | corr=5 cove=5 cons=5 grou=5 rele=5 | avg=5.00 | PASS | 21s | elapse

## 10. Inspecting the results (optional)

Aggregate scores per run configuration straight from the judge database.

In [ ]:
import sqlite3
import pandas as pd

with sqlite3.connect(JUDGE_DB_PATH) as conn:
    summary = pd.read_sql_query(
        """
        SELECT rag_algorithm, top_k, reranking_method,
               COUNT(*)            AS n,
               ROUND(AVG(correctness), 2)  AS correctness,
               ROUND(AVG(coverage), 2)     AS coverage,
               ROUND(AVG(consistency), 2)  AS consistency,
               ROUND(AVG(groundedness), 2) AS groundedness,
               ROUND(AVG(relevance), 2)    AS relevance,
               ROUND(AVG(success), 3)      AS pass_rate
        FROM evaluation_results
        GROUP BY rag_algorithm, top_k, reranking_method
        """,
        conn,
    )

summary

,rag_algorithm,top_k,reranking_method,n,correctness,coverage,consistency,groundedness,relevance,pass_rate
0,"BM25, Dense Vector Search",15,RRF,100,4.04,3.77,4.4,4.38,4.42,0.88
